2115. Find All Possible Recipes from Given Supplies
Medium

You have information about n different recipes. You are given a string array recipes and a 2D string array ingredients. The ith recipe has the name recipes[i], and you can create it if you have all the needed ingredients from ingredients[i]. A recipe can also be an ingredient for other recipes, i.e., ingredients[i] may contain a string that is in recipes.

You are also given a string array supplies containing all the ingredients that you initially have, and you have an infinite supply of all of them.

Return a list of all the recipes that you can create. You may return the answer in any order.

Note that two recipes may contain each other in their ingredients.

Example 1:

Input: recipes = ["bread"], ingredients = [["yeast","flour"]], supplies = ["yeast","flour","corn"]
Output: ["bread"]
Explanation:
We can create "bread" since we have the ingredients "yeast" and "flour".

Example 2:

Input: recipes = ["bread","sandwich"], ingredients = [["yeast","flour"],["bread","meat"]], supplies = ["yeast","flour","meat"]
Output: ["bread","sandwich"]
Explanation:
We can create "bread" since we have the ingredients "yeast" and "flour".
We can create "sandwich" since we have the ingredient "meat" and can create the ingredient "bread".

Example 3:

Input: recipes = ["bread","sandwich","burger"], ingredients = [["yeast","flour"],["bread","meat"],["sandwich","meat","bread"]], supplies = ["yeast","flour","meat"]
Output: ["bread","sandwich","burger"]
Explanation:
We can create "bread" since we have the ingredients "yeast" and "flour".
We can create "sandwich" since we have the ingredient "meat" and can create the ingredient "bread".
We can create "burger" since we have the ingredient "meat" and can create the ingredients "bread" and "sandwich".


### Analysis 

You can make a recipe only when all its ingredients are available. A completed recipe can become an ingredient for another recipe. Supplies are unlimited, and you return every recipe you can make, in any order. Recipes may contain circular dependencies

recipes = ["dough", "pizza", "soup"]

ingredients = [
    ["flour", "water"],    # dough
    ["dough", "cheese"],   # pizza
    ["water", "carrot"]    # soup
]

supplies = ["flour", "water", "cheese"]

We can make:
1. dough, using flour and water.
2. pizza, using dough and cheese.
3. We cannot make soup, because carrot is unavailable.

Naive Approach: Repeatedly Scan all recepies 

Maintain a set of available items, initially containing the supplies.

Repeatedly:
1. Check every unmade recipe.
2. If all its ingredients are available, make it.
3. Add its name to the available set.
4. Stop when an entire pass produces no new recipes.
This is correct. However, a long dependency chain listed in reverse order may unlock only one recipe per pass, causing repeated checks.

Let:
- R = number of recipes.
- I = total number of ingredient entries across all recipes.
- S = number of supplies.
Under this problem’s nonempty ingredient lists:
- Time: O(S + R × I) worst case.
- Extra space: O(S + R).
We can improve this by updating only recipes affected by a newly available ingredient.

Optimal approach: dependency counts + queue

Build edges in this direction:
ingredient → recipes that need it

For our example:

dependents = {
    "flour": ["dough"],
    "water": ["dough", "soup"],
    "dough": ["pizza"],
    "cheese": ["pizza"],
    "carrot": ["soup"]
}

Track how many ingredients each recipe still needs:
remaining = {
    "dough": 2,
    "pizza": 2,
    "soup": 2
}

Start the queue with the supplies.

Whenever an item is removed from the queue:
1. Find recipes that require it.
2. Decrease their remaining ingredient counts.
3. When a count reaches zero, that recipe can be made.
4. Add the recipe to both the result and the queue.
Adding it to the queue allows it to unlock further recipes.

Important difference from Course Schedule

We must not start with every graph node having no incoming edges.

For example, "carrot" has no prerequisites represented in this graph, but that does not mean we possess it. Only the given supplies are initially available.


In [5]:
# Optimal Solution 

from collections import defaultdict, deque


class Solution:
    def findAllRecipes(
        self,
        recipes: list[str],
        ingredients: list[list[str]],
        supplies: list[str]
    ) -> list[str]:

        dependents = defaultdict(list)
        remaining = {}

        for recipe, required in zip(recipes, ingredients):
            # Initially, no ingredient availability has been processed.
            remaining[recipe] = len(required)

            for ingredient in required:
                dependents[ingredient].append(recipe)

        # Only supplied items are initially available.
        queue = deque(supplies)
        result = []

        while queue:
            item = queue.popleft()

            # Update only recipes that need this available item.
            for recipe in dependents.get(item, []):
                remaining[recipe] -= 1

                if remaining[recipe] == 0:
                    result.append(recipe)

                    # This recipe can now serve as an ingredient.
                    queue.append(recipe)

        return result

# Usage
recipes = ["bread"]
ingredients = [["yeast","flour"]]
supplies = ["yeast","flour","corn"]
# Output: ["bread"]
soln = Solution()
result = soln.findAllRecipes(recipes, ingredients, supplies)
print(result)

# ---another example 
recipes = ["bread","sandwich","burger", "soup"]
ingredients = [["yeast","flour"],["bread","meat"],["sandwich","meat","bread"], ["flour", "carrot"]]
supplies = ["yeast","flour","meat"]
# Output: ["bread","sandwich","burger"]
result = soln.findAllRecipes(recipes, ingredients, supplies)
print(result)


['bread']
['bread', 'sandwich', 'burger']
